<!-- nav -->
[← 1 · Constraints your data must keep](01_Constraints_Your_Data_Must_Keep.ipynb) · [Home](../../README.md) · [3 · Links as constraints →](03_Links_As_Constraints.ipynb)

# 2 · Finding what matches

**The problem.** The same constraints that check data also *select* it: "the adults", "each contact with each of their
numbered phones". Writing those selections again as loops repeats the constraints, and repeats their mistakes. A
predicate is already a query: its matches are what it selects.

This case study runs predicates as queries over a store, and looks at how the query is planned.

In [1]:
import { Expressions as E } from "@mbse/expressions";
import { Predicates, Queries } from "@mbse/patterns";
import { Contact, Phone, book, listed } from "./toolkit.ts";

const store = book();
const ann = store.Contact().name("Ann").age(34n)
  .phones((e: any) => e.label("home").phone((p: any) => p.number("555-0100")))
  .phones((e: any) => e.label("work").phone((p: any) => p.number("555-0199"))).create();
const bob = store.Contact().name("Bob").phones((e: any) => e.label("work").phone((p: any) => p)).create();
const kid = store.Contact().name("Kid").age(9n).create();
listed(store, ann, bob, kid);
const stray = store.Contact().name("Stray").age(50n).create(); // built, but never listed
const query = new Queries.Scan(store);
const the = E.variable("the");

## Step 1: a predicate's matches

`new Queries.Scan(store)` makes any store queryable. `select` streams the matches for which the constraint holds, each a
record of symbols to objects:

In [2]:
const IsAnAdult = new Predicates.OfPredicate.Builder().name("IsAnAdult").symbols({ the: Contact }).requires(the.age.ge(18n)).create();
console.log([...query.select(IsAnAdult)].map((m: any) => m.the.name));

[ "Ann" ]


Only Ann. Bob's age is unknown, so he isn't selected unless you ask (`unknown`, the third argument). The stray contact
isn't selected either: a query sees the store's **data**, what its singleton directory reaches, and the stray was never
listed.

In [3]:
console.log([...query.select(IsAnAdult, null, true)].map((m: any) => m.the.name));

[ "Ann", "Bob" ]


`select` returns a generator, and reads the store only as you ask for matches: the first `next()` does the work.

## Step 2: variables

A constraint can name values it doesn't bind, **variables**, which the query gives:

In [4]:
const Older = new Predicates.OfPredicate.Builder().name("Older").symbols({ the: Contact }).requires(the.age.ge(E.variable("least"))).create();
console.log([...query.select(Older, { least: 5n })].map((m: any) => m.the.name));

[ "Ann", "Kid" ]


## Step 3: two symbols, and the plan

With two symbols, the matches are every combination of a contact and a phone, filtered by the constraint. Scanning every
combination would be slow; the constraint's shape says what's practical. `explain` shows the plan:

In [5]:
const [c, p] = [E.variable("c"), E.variable("p")];
const Numbered = new Predicates.OfPredicate.Builder().name("Numbered").symbols({ c: Contact, p: Phone })
  .requires(Predicates.Contains(c.phones, (e) => e.phone.eq(p))).requires(p.has("number")).create();
console.log(query.explain(Numbered));
console.log([...query.select(Numbered)].map((m: any) => [m.c.name, m.p.number]));

[ "c: scan Contact", "p: c.phones to phone, any number, then 2 tests" ]


[ [ "Ann", "555-0100" ], [ "Ann", "555-0199" ] ]


The `Contains` conjunct relates `p` to `c` through the relation, so the plan takes `p` from `c`'s entries (a **hop**)
instead of scanning every phone. Each `.requires(...)` is a conjunct, tested as soon as the symbols it reads are bound.

The relation's `unique("owner", "label")` (in `toolkit.ts`) says a phone determines its one entry, so a hop from a phone
to its owner gives **at most one** contact. The planner uses that to order the hops:

In [6]:
const Owner = new Predicates.OfPredicate.Builder().name("Owner").symbols({ c: Contact, p: Phone })
  .requires(Predicates.Contains(p.owners, (e) => e.owner.eq(c))).create();
console.log(query.explain(Owner));

[ "p: scan Phone", "c: p.owners to owner, at most one, then 1 test" ]


## Why it works this way

- **One constraint, two uses.** Checking and selecting are the same constraint read two ways, so they can't disagree.
- **The constraint's shape is the hint.** You don't annotate a query for performance. A relation between symbols, said
  as a `Contains`, is a hop; a relation's `unique` clause says how far it fans out. `explain` shows what was made of it.
- **Lazy, over the store's data.** A query reads as you iterate, and sees what the store's roots reach, the same
  data a snapshot would save.

<!-- nav -->
[← 1 · Constraints your data must keep](01_Constraints_Your_Data_Must_Keep.ipynb) · [Home](../../README.md) · [3 · Links as constraints →](03_Links_As_Constraints.ipynb)